[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/block3.ipynb)

# Block 3: a small network, and a model that cheated

Nineteen minutes, together. Every answer is written. Read, run, look.

## 1. Images that ship with scikit-learn

1797 handwritten digits, each one eight pixels by eight. This is the data you
could not write a rule for in block 1.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
print("rows and columns:", digits.data.shape)

fig, ax = plt.subplots(1, 8, figsize=(9, 1.6))
for i in range(8):
    ax[i].imshow(digits.images[i], cmap="gray_r")
    ax[i].set_title(str(digits.target[i]))
    ax[i].axis("off")

## 2. One small network, trained

One hidden layer of sixteen neurons. Watch the number fall as it trains: that
is the loss, and making it small is the whole of what training does.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=0)

net = MLPClassifier(hidden_layer_sizes=(16,), max_iter=400,
                    random_state=0).fit(Xtr, ytr)

print("held back:", round(net.score(Xte, yte), 3))

plt.figure(figsize=(5, 2.5))
plt.plot(net.loss_curve_, color="#1f3864")
plt.title("the loss, falling")
plt.xlabel("how many times it went round the training loop")

Sixteen neurons and four hundred times round the loop. Nobody wrote a rule for
the shape of a seven.

## 3. The same network, on sensor readings

Thirteen measurements from an instrument, not an image. Same three lines. The
network does not know or care which kind of data it got.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine()
Xtr, Xte, ytr, yte = train_test_split(
    wine.data, wine.target, test_size=0.3, random_state=0)

net2 = MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000,
                     random_state=0).fit(Xtr, ytr)
print("held back:", round(net2.score(Xte, yte), 3))

## 4. Now a column that should not be there

A colleague sends you the same table with one extra column: the batch number
the sample was processed in. It looks harmless. Fit the same network.

In [ ]:
import numpy as np

# The samples were processed one kind at a time, so the batch number happens
# to follow the answer. Nobody did this on purpose. That is the point.
batch = wine.target * 100 + np.arange(len(wine.target)) % 7

leaky = np.column_stack([wine.data, batch])

Xtr, Xte, ytr, yte = train_test_split(
    leaky, wine.target, test_size=0.3, random_state=0)

net3 = MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000,
                     random_state=0).fit(Xtr, ytr)
print("held back, with the extra column:", round(net3.score(Xte, yte), 3))

## 5. What just happened

The score went up. It went up because the extra column contains the answer:
divide the batch number by a hundred and you have the label.

The network is not clever. It is reading the answer off the table.

Nothing in the code failed. Nothing warned you. The held back rows have the
same bad column in them, so the check you were relying on is broken in exactly
the same way.

In [ ]:
print("batch numbers for each kind:")
for k in sorted(set(wine.target)):
    print("  kind", k, "->", sorted(set(batch[wine.target == k]))[:4], "...")

This is data leakage, and the giveaway is a score that looks too good rather
than a score that looks bad.

When a number surprises you upwards, that is when to go and look at the
columns.